# LLM-as-a-Judge example

Walks the `inference.judges` package over a tiny product-review dataset.

Architecture:
- One LLM call per (subject, judge) row. No extraction fallback.
- The model reasons inline and ends its reply with `<final_answer>LABEL</final_answer>` where `LABEL` is verbatim one of the configured classes (or `__NONE__` as a last resort). Parsing is case-sensitive.
- Per-judge async queues with configurable worker concurrency (see section 2).
- Resume by default: successful rows in the CSV are skipped on re-run; failed rows are retried.
- Optional hidden reasoning via `thinking_budget_tokens` (section 5).

## 0. Setup

Needs `OPENROUTER_API_KEY` in `.env`. Uses `config/judge_smoke.yaml` (free OpenRouter models). Swap in your own config for paid runs.

In [1]:
from pathlib import Path

import nest_asyncio
import pandas as pd

nest_asyncio.apply()

from inference import (
    JudgeConfig,
    JudgeExecutionConfig,
    create_client,
    run_judges,
    subjects_from_dataframe,
)


def repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "config").is_dir() and (p / "src" / "inference").is_dir():
            return p
    return Path.cwd()


ROOT = repo_root()
CONFIG_PATH = ROOT / "config" / "judge_smoke.yaml"
OUTPUT_DIR = ROOT / "logs" / "judges" / "example"

client = create_client(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Output dir:", OUTPUT_DIR)

Config: /home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/config/judge_smoke.yaml
Output dir: /home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example


## 1. Input dataset

Ten short product reviews with gold labels. The labels live in metadata and surface in the output dataframe as `metadata_true_label`.

In [2]:
reviews_df = pd.DataFrame(
    [
        # id, review_text, true_label
        ("rev-01", "Best purchase of the year — genuinely impressed with the build quality.", "Positive"),
        ("rev-02", "I use it every day and it just keeps working. No complaints.", "Positive"),
        ("rev-03", "Customer support resolved my issue in under five minutes.", "Positive"),
        ("rev-04", "Stopped charging after a month. Save your money.", "Negative"),
        ("rev-05", "Doesn't match the photos — feels like false advertising.", "Negative"),
        ("rev-06", "Returned it the same day. Completely useless.", "Negative"),
        ("rev-07", "Solid hardware, but the battery life is unacceptable for travel.", "Mixed"),
        ("rev-08", "It does what it says. Nothing special, nothing terrible.", "Mixed"),
        ("rev-09", "Honestly exceeded my expectations — setup took ten minutes.", "Positive"),
        ("rev-10", "Worst customer service experience I have had in years.", "Negative"),
    ],
    columns=["review_id", "review_text", "true_label"],
)

subjects = subjects_from_dataframe(
    reviews_df,
    id_field="review_id",
    content_field="review_text",
    metadata_fields=["true_label"],
    source_id="notebook-reviews-v1",
)

print(f"{len(subjects)} subjects ready")
reviews_df

10 subjects ready


,review_id,review_text,true_label
0,rev-01,Best purchase of the year — genuinely impresse...,Positive
1,rev-02,I use it every day and it just keeps working. ...,Positive
2,rev-03,Customer support resolved my issue in under fi...,Positive
3,rev-04,Stopped charging after a month. Save your money.,Negative
4,rev-05,Doesn't match the photos — feels like false ad...,Negative
5,rev-06,Returned it the same day. Completely useless.,Negative
6,rev-07,"Solid hardware, but the battery life is unacce...",Mixed
7,rev-08,"It does what it says. Nothing special, nothing...",Mixed
8,rev-09,Honestly exceeded my expectations — setup took...,Positive
9,rev-10,Worst customer service experience I have had i...,Negative


## 2. Configure judges and run

Each (subject, judge) pair triggers exactly one LLM call. The class block and sentinel rules are appended automatically when `classes` is set, so the rubric in `judge_prompt` should focus on the evaluation task itself.

This demo uses a single judge (`gpt-oss-20b-free`) because OpenRouter's free tier has retired most other free endpoints. `judges` accepts a list — add aliases there to run several judges side by side; each gets its own async queue, so a flaky one cannot block the rest.

Common knobs:
- `judges` — list of model aliases from your inference config. Each runs on its own async queue.
- `classes` — allowed labels. Pass `None` for free-form judging (section 7).
- `max_tokens` — cap on the reply. Leave headroom for inline reasoning plus the sentinel line.
- `temperature` — 0.0 for reproducibility.
- `thinking_budget_tokens` — hidden reasoning budget; see section 5.
- `log_verbosity` — `silent`, `normal`, `verbose`, `debug`. Logs go to stderr.
- `resume=True` (default) — re-running this cell skips rows already marked `success`.

Concurrency lives on `JudgeExecutionConfig`:
- `default_workers` — workers per judge queue. Each worker holds one in-flight call, so N workers ≈ N concurrent requests against that model.
- `per_judge_workers` — override per alias when one model can take more parallelism than another.
- `call_timeout_s` — per-call timeout.

Keep `default_workers=1` on free-tier OpenRouter (10 rpm). On paid or self-hosted endpoints bump it to 4–8 per judge and watch throughput climb.

In [3]:
FRESH_RUN = False  # flip to True to wipe prior judgments for this experiment

config = JudgeConfig(
    experiment_name="notebook-sentiment",
    judges=["gpt-oss-20b-free"],
    judge_prompt=(
        "You are a sentiment classifier for short product reviews. "
        "Decide whether the overall sentiment is Positive, Negative, or Mixed. "
        "Reason briefly about tone, recommendation, and any qualifiers before answering."
    ),
    classes=["Positive", "Negative", "Mixed"],
    output_dir=OUTPUT_DIR,
    max_tokens=400,
    temperature=0.0,
    # thinking_budget_tokens=None,   # off by default; see section 5
    log_verbosity="verbose",
)

if FRESH_RUN:
    csv_guess = OUTPUT_DIR / f"{config.experiment_name}.judgments.csv"
    if csv_guess.exists():
        csv_guess.unlink()
        print("Deleted prior CSV:", csv_guess)

# 1 worker per judge for free-tier rate limits. Bump default_workers (or use
# per_judge_workers={"alias": N}) when running against paid endpoints.
execution = JudgeExecutionConfig(
    call_timeout_s=90.0,
    default_workers=1,
    # per_judge_workers={"gpt-oss-20b-free": 4},
)

# Progress logs appear on stderr while this cell runs.
result = await run_judges(client, subjects, config, execution)

import sys
print("CSV:", result.csv_path); sys.stdout.flush()
print("Summary:"); sys.stdout.flush()
for k, v in result.summary.items():
    print(f"  {k}: {v}")

judge: starting run experiment='notebook-sentiment' judges=['gpt-oss-20b-free'] subjects=10 cfg_hash=ea6190c9c3ad


judge: csv=/home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-sentiment.judgments.csv


judge: gpt-oss-20b-free | workers=1 pending=10


judge: gpt-oss-20b-free | [  1/10 ] ok  rev-01               -> Positive             (13.4s 366tok)


judge: gpt-oss-20b-free | [  2/10 ] FAIL rev-02               InferenceRequestError: Inference failed for alias='gpt-oss-20b-free', provider='openrouter', model='openai/gpt-oss-20... (12.2s)


...

judge: gpt-oss-20b-free | DONE  ok=8 miss=0 fail=2 elapsed=2m22s


judge: SUMMARY ok=8 fail=2 skipped_resume=0 elapsed=2m22s


CSV: /home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-sentiment.judgments.csv


Summary:


  total_subjects: 10
  total_judges: 1
  skipped_resume: 0
  per_judge: {'gpt-oss-20b-free': {'completed': 8, 'classification_failed': 0, 'call_failed': 2, 'skipped_resume': 0}}
  csv_path: /home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-sentiment.judgments.csv
  adapter_summary: {'total': 10, 'emitted': 10}


## 3. Output dataframe

`result.dataframe` has one row per (subject, judge). Useful columns:

- `status` — `success`, `classification_failed`, `call_failed`.
- `parse_status` — `matched`, `none_declared`, `unmatched` (sentinel found but contents not in `classes`), `missing_sentinel` (no tag at all), or `free_form`.
- `final_class` — the parsed label, set only when `parse_status == matched`.
- `raw_output` — the full model reply (reasoning plus sentinel).

In [4]:
df = result.dataframe

display_cols = [
    "subject_id",
    "judge_alias",
    "final_class",
    "parse_status",
    "status",
    "metadata_true_label",
    "latency_ms",
    "total_tokens",
]
show = [c for c in display_cols if c in df.columns]
df[show].sort_values(["subject_id", "judge_alias"]).reset_index(drop=True)

,subject_id,judge_alias,final_class,parse_status,status,metadata_true_label,latency_ms,total_tokens
0,rev-01,gpt-oss-20b-free,Positive,matched,success,Positive,13442.946,366.0
1,rev-02,gpt-oss-20b-free,NaN,missing_sentinel,call_failed,Positive,12201.645,NaN
2,rev-03,gpt-oss-20b-free,NaN,missing_sentinel,call_failed,Positive,8704.630,NaN
3,rev-04,gpt-oss-20b-free,Negative,matched,success,Negative,8108.622,363.0
4,rev-05,gpt-oss-20b-free,Negative,matched,success,Negative,10679.029,364.0
5,rev-06,gpt-oss-20b-free,Negative,matched,success,Negative,19946.859,362.0
6,rev-07,gpt-oss-20b-free,Mixed,matched,success,Mixed,12745.930,371.0
7,rev-08,gpt-oss-20b-free,Mixed,matched,success,Mixed,14206.247,408.0
8,rev-09,gpt-oss-20b-free,Positive,matched,success,Positive,13717.126,363.0
9,rev-10,gpt-oss-20b-free,Negative,matched,success,Negative,14734.698,363.0


### 3a. Accuracy vs. gold labels

Restricted to rows that returned a valid class.

In [5]:
ok = df[df["status"] == "success"].copy()
if ok.empty:
    print("No successful judgments yet — likely rate-limited. Re-run section 2.")
else:
    ok["correct"] = ok["final_class"] == ok["metadata_true_label"]
    acc = (
        ok.groupby("judge_alias")["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
    )
    display(acc)

    disagreements = ok[~ok["correct"]][
        ["subject_id", "judge_alias", "final_class", "metadata_true_label"]
    ]
    if disagreements.empty:
        print("All successful judgments match gold labels.")
    else:
        print("Disagreements:")
        display(disagreements)

,accuracy,n
judge_alias,,
gpt-oss-20b-free,1.0,8


All successful judgments match gold labels.


### 3b. One full reasoning trace

In [6]:
ok = df[df["status"] == "success"]
if ok.empty:
    print("No successful judgments to show.")
else:
    row = ok.iloc[0]
    print(f"[{row['judge_alias']}] {row['subject_id']} -> {row['final_class']!r}")
    print("---")
    print(row["raw_output"])

[gpt-oss-20b-free] rev-01 -> 'Positive'
---
<final_answer>Positive</final_answer>


## 4. Resume

Re-running with the same `JudgeConfig` skips any row already marked `success` in the CSV. Rows with `call_failed` or `classification_failed` are retried, which is what you want after a rate-limit burst or a sentinel miss.

In [7]:
_prev = pd.read_csv(result.csv_path)
_done = int((_prev["status"] == "success").sum())
print(f"Already successful (will skip): {_done} / {len(_prev)} rows")

result_resume = await run_judges(client, subjects, config, execution)
print("skipped_resume:", result_resume.summary.get("skipped_resume"))
print("per_judge:", result_resume.summary.get("per_judge"))

judge: starting run experiment='notebook-sentiment' judges=['gpt-oss-20b-free'] subjects=10 cfg_hash=ea6190c9c3ad


judge: csv=/home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-sentiment.judgments.csv


judge: resume -> 8 (subject, judge) cells already done


judge: gpt-oss-20b-free | workers=1 pending=2


Already successful (will skip): 8 / 10 rows


judge: gpt-oss-20b-free | [  1/2  ] ok  rev-02               -> Positive             (11.0s 367tok)


judge: gpt-oss-20b-free | [  2/2  ] ok  rev-03               -> Positive             (19.6s 363tok)


judge: gpt-oss-20b-free | DONE  ok=2 miss=0 fail=0 elapsed=42.4s


judge: SUMMARY ok=2 fail=0 skipped_resume=8 elapsed=42.4s


skipped_resume: 8
per_judge: {'gpt-oss-20b-free': {'completed': 2, 'classification_failed': 0, 'call_failed': 0, 'skipped_resume': 0}}


## 5. Hidden reasoning (opt-in)

Set `thinking_budget_tokens=N` to let the model do hidden reasoning before producing the sentinel. The runner forwards `thinking={"type": "enabled", "budget_tokens": N}` to the provider via LiteLLM.

- Anthropic (extended thinking) and Gemini (thinking) honor the kwarg. Other providers ignore it, so it is safe to leave on for mixed-provider runs.
- `raw_output` still contains the final reply only; hidden thinking tokens are not returned.
- `thinking_budget_tokens` participates in `judge_config_hash`, so changing it invalidates the resume cache. We use a different `experiment_name` here to keep the two runs in separate CSVs.
- Reasonable starting points: 1024 for moderate help, 2048–4096 for harder rubrics.

In [8]:
thinking_cfg = JudgeConfig(
    experiment_name="notebook-sentiment-thinking",
    judges=["gpt-oss-20b-free"],
    judge_prompt=(
        "You are a sentiment classifier for short product reviews. "
        "Decide whether the overall sentiment is Positive, Negative, or Mixed. "
        "Weigh praise, complaints, and qualifiers carefully before answering."
    ),
    classes=["Positive", "Negative", "Mixed"],
    output_dir=OUTPUT_DIR,
    max_tokens=400,
    temperature=0.0,
    thinking_budget_tokens=2048,
    log_verbosity="normal",
)

thinking_result = await run_judges(client, subjects, thinking_cfg, execution)

tdf = thinking_result.dataframe
show_cols = [c for c in [
    "subject_id", "judge_alias", "final_class", "parse_status",
    "status", "metadata_true_label", "latency_ms", "total_tokens",
] if c in tdf.columns]
tdf[show_cols].sort_values(["subject_id", "judge_alias"]).reset_index(drop=True)

judge: starting run experiment='notebook-sentiment-thinking' judges=['gpt-oss-20b-free'] subjects=10 cfg_hash=6665e4a17421


judge: csv=/home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-sentiment-thinking.judgments.csv


judge: gpt-oss-20b-free | workers=1 pending=10


judge: gpt-oss-20b-free | DONE  ok=10 miss=0 fail=0 elapsed=2m40s


judge: SUMMARY ok=10 fail=0 skipped_resume=0 elapsed=2m40s


,subject_id,judge_alias,final_class,parse_status,status,metadata_true_label,latency_ms,total_tokens
0,rev-01,gpt-oss-20b-free,Positive,matched,success,Positive,12042.223,365
1,rev-02,gpt-oss-20b-free,Positive,matched,success,Positive,10534.816,366
2,rev-03,gpt-oss-20b-free,Positive,matched,success,Positive,15993.948,362
3,rev-04,gpt-oss-20b-free,Negative,matched,success,Negative,16932.641,362
4,rev-05,gpt-oss-20b-free,Negative,matched,success,Negative,9696.523,363
5,rev-06,gpt-oss-20b-free,Negative,matched,success,Negative,6981.674,361
6,rev-07,gpt-oss-20b-free,Mixed,matched,success,Mixed,9243.207,370
7,rev-08,gpt-oss-20b-free,Mixed,matched,success,Mixed,12553.173,379
8,rev-09,gpt-oss-20b-free,Positive,matched,success,Positive,11344.356,362
9,rev-10,gpt-oss-20b-free,Negative,matched,success,Negative,13453.387,362


## 6. Logging levels

| Level | Output |
|-------|--------|
| `silent` | nothing |
| `normal` | start, per-judge done, final summary |
| `verbose` | every row outcome plus progress counters |
| `debug` | latency, tokens, queue details |

All logs go to stderr; in Jupyter that appears under the running cell.

## 7. Free-form mode

Drop `classes` for open-ended replies — no sentinel parsing. `final_class` is `None` and successful rows get `parse_status=free_form`.

In [9]:
freeform_cfg = JudgeConfig(
    experiment_name="notebook-rubric",
    judges=["gpt-oss-20b-free"],
    judge_prompt=(
        "Rate this review on a 1–5 usefulness scale and explain why in 2–3 sentences."
    ),
    classes=None,
    output_dir=OUTPUT_DIR,
    max_tokens=300,
    log_verbosity="normal",
)

freeform_subjects = subjects_from_dataframe(
    reviews_df.head(2),
    id_field="review_id",
    content_field="review_text",
)

freeform_result = await run_judges(client, freeform_subjects, freeform_cfg)
freeform_result.dataframe[
    ["subject_id", "judge_alias", "final_class", "status", "raw_output"]
]

judge: starting run experiment='notebook-rubric' judges=['gpt-oss-20b-free'] subjects=2 cfg_hash=bf05165dfcce


judge: csv=/home/lukas/projects/Investigating-Personalization-Effects-in-LLMs/logs/judges/example/notebook-rubric.judgments.csv


judge: gpt-oss-20b-free | workers=1 pending=2


judge: gpt-oss-20b-free | DONE  ok=2 miss=0 fail=0 elapsed=36.7s


judge: SUMMARY ok=2 fail=0 skipped_resume=0 elapsed=36.7s


,subject_id,judge_alias,final_class,status,raw_output
0,rev-01,gpt-oss-20b-free,None,success,**Usefulness rating: 2**\n\nThe comment is ver...
1,rev-02,gpt-oss-20b-free,None,success,**Usefulness rating: 2/5**\n\nThe comment is e...
